# **Scraping Planos Ultranet**

_Página do provedor: https://ultra.psi.br/home/_

In [1]:
# Bibliotecas
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date

## **1. Extração**

In [3]:
# Obter lista de planos
texto_html = open("../../data/scraped/ultranet.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
container_planos = soup.find("div", id="painel-planos")
planos = container_planos.find_all("article")
print(f"{len(planos)} cartões de planos capturados.")

planos

4 cartões de planos capturados.


[<article class="relative flex flex-col gap-5 rounded-[22px] px-[22px] py-7 border border-borda bg-white text-texto w-[290px] max-w-[85vw] shrink-0 snap-start md:w-auto md:max-w-none"><!-- --><div class="flex flex-col gap-1.5"><h3 class="text-noite flex items-baseline gap-1.5"><span class="text-[46px] leading-none font-bold tracking-[-0.03em] md:text-[50px]">400</span><span class="text-lg font-semibold">Mega</span></h3><p class="text-texto-2 text-[15px]">Essencial para sua rotina online</p></div><div><!-- --><p class="text-noite flex items-baseline gap-[3px]"><span class="sr-only">R$ 109,90 por mês</span><span aria-hidden="true" class="text-[15px] font-semibold">R$</span><span aria-hidden="true" class="text-[34px] font-titulo leading-none font-bold">109</span><span aria-hidden="true" class="text-[15px] font-semibold"> ,90</span><span aria-hidden="true" class="text-sm text-texto-2"> /mês </span></p></div><ul class="flex flex-col gap-2.5 text-[15px]"><!--[--><li class="flex items-center 

In [15]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = plano.find_all("h3")[0].text
    preco = plano.find_all("p")[1].text
    svas = plano.find_all("p")[2].text

    return {
        "velocidade": velocidade or None,
        "preco": preco or None,
        "svas": svas
    }

df = pd.DataFrame(raspar_plano(p) for p in planos).drop_duplicates().reset_index(drop=True)
df

,velocidade,preco,svas
0,400Mega,"R$ 109,90 por mêsR$109 ,90 /mês",Apps: 1 app Standard à escolha
1,500Mega,"R$ 119,90 por mêsR$119 ,90 /mês",Apps: 2 apps Standard à escolha
2,700Mega,"R$ 129,90 por mêsR$129 ,90 /mês",Apps: Telecine ou Globoplay + 2 apps Standard
3,900Mega,"R$ 149,90 por mêsR$149 ,90 /mês",Apps: Telecine ou Globoplay + 3 apps Standard


## **2. Tratamento**

In [16]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Ultranet"

# Velocidade
df_tratado["VELOCIDADE"] = df_tratado["velocidade"].str.extract(r"(\d+)")

# SVAs
df_tratado["SVAs"] = df_tratado["svas"]

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

,DATA,ISP,VELOCIDADE,SVAs,PRECO
0,2026-10-22,Ultranet,400,Apps: 1 app Standard à escolha,109.9
1,2026-10-22,Ultranet,500,Apps: 2 apps Standard à escolha,119.9
2,2026-10-22,Ultranet,700,Apps: Telecine ou Globoplay + 2 apps Standard,129.9
3,2026-10-22,Ultranet,900,Apps: Telecine ou Globoplay + 3 apps Standard,149.9


## **3. Carga**

In [17]:
# Salvando
provedor = "ULTRANET.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")

🟢 2026-10-22-ULTRANET.csv salvo!
